<a href="https://colab.research.google.com/github/d-noe/NLP_DH_F26/blob/main/code/1_introduction/Tutorial_1_Annotation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tutorial Session 1: Text Annotation & Inter-rater Agreement

![](https://external-content.duckduckgo.com/iu/?u=https%3A%2F%2Fwww.artribune.com%2Fwp-content%2Fuploads%2F2017%2F11%2FRen%25C3%25A9-Magritte-La-Trahison-des-images-Ceci-n%25E2%2580%2599est-pas-une-pipe.jpg&f=1&nofb=1&ipt=62dcbdb91fec90d0cd0276ee37c0739958ceec8979336db20874b8c646889f86$0)
<p align="right">
  <i>La Trahison des Images</i>. René Magritte (1928-1929). Oil on canvas.
</p>

This notebook contains the implementation for a workshop on **sentiment annotation**.

First, you will be directly prompted to label the *sentiment* of a reduced sample of sentences ($N=30$) into one of the four defined categories: `Positive`, `Negative`, `Mixed`, or `Neutral`.

Then, the annotation of all of the participants will be grouped to compute aggregate measures of inter-annotator agreement (IAA), as well as detailed annotation statistics per sample, prompting a discussion on this annotation task.


## Let's start by preparing the machinery

### 1. Python packages

We first need to install the dependencies that we will be using later on in this notebook:

- [`pigeonxt-jupyter`](https://pypi.org/project/pigeonxt-jupyter/$0): An extension of [`🐦 pigeon`](https://github.com/agermanidis/pigeon$0), a simple widget to quickly annotate a dataset of unlabeled examples directly within a Jupyter notebook.
- [statsmodels](https://www.statsmodels.org/stable/index.html$0): a Python package for statistical computations, used here to compute Fleiss' kappa.
- [krippendorff](https://github.com/pln-fing-udelar/fast-krippendorff$0): a fast implementation of the computation of Krippendorff's alpha

In [ ]:
!pip install pigeonxt-jupyter==0.4.1
!pip install krippendorff statsmodels

In [ ]:
# Now that they are installed,
# we can import the package to annotate data: pigeonXT
# as well as standard Python libraries (pre-installed in Colab)

import pandas as pd # to load and manipulate the data
import pigeonXT as pixt # to annotate samples directly in the notebook

### 2. Loading the data

The [GitHub repository](https://github.com/d-noe/NLP_DH_F26) for the class already contains some helper functions, so we'll get the [`helpers.py`](https://github.com/d-noe/NLP_DH_F26/blob/main/code/scripts/helpers.py) with `wget` (download locally; on Colab, stored in `Files` and deleted with the session if not mounted on Drive).  

We can then use the `load_csv_from_github` function to load a `csv` file by simply giving the path in the repository, here: [data/annotation_data/sentences_to_annot.csv](https://github.com/d-noe/NLP_DH_F26/blob/main/data/annotation_data/sentences_to_annot.csv).

In [ ]:
!wget https://raw.githubusercontent.com/d-noe/NLP_DH_F26/refs/heads/main/code/scripts/helpers.py
from helpers import load_csv_from_github # load function from helpers.py

# Load the data
df = load_csv_from_github("data/annotation_data/sentences_to_annot.csv")
print(f"The DataFrame contains {len(df)} rows.")

# Display 5 random rows
df.sample(5)


The `df` DataFrame that we have just loaded contains the following columns:

- `Sentence_id`: a unique identifier for each of the sample
- `Sentence`: the raw text that we will need to label

### 3. Converting the data

Now, to be able to use [`🐦 pigeon`](https://github.com/agermanidis/pigeon) annotation widget, we will convert our data into a `pigeon`-compliant format: using a list of samples, and providing the predefined categories.

In [ ]:
# cast the samples into a list
sentences = df['Sentence'].tolist()

In [ ]:
# the list of predefined labels
categories = ['Positive', 'Negative', 'Mixed', 'Neutral']

## Let's annotate!

Now that everything is ready, we can start annotating the samples.

When running the following cell, you will see the four categories, and the sample will be printed below. When clicking on a button associated with a category, it will directly assign the chosen label to the current sample, and go to the next sample, untill reaching completion (30/30).

**WARNING**: you cannot undo your annotation, be sure of your choice when clicking the button.

The annotation manual is the following:

> Please read the instructions for the sentiment labelling task.
>
> You will see different sentences in the notebook. Your task is to decide whether each sentence has a *positive*, *negative*, *neutral* or *mixed* sentiment.
For example, “*I like warm summer evenings*” has a *positive* sentiment.
>
> “*Mixed*” label refers to a sentence that has a mixed sentiment. For example, the sentence “*I liked the food, but the waiter was rude*” has a *mixed* sentiment.
>
> Try to be objective with your judgement.

In [ ]:
# When running this cell, you will

annotations = pixt.annotate(
    sentences,
    options=categories,
    include_skip=False
)

Now that we have annotated **all** of the samples, we will convert them back to a DataFrame and save them as `.csv`.

We will have a look at the annotation in the cell below, please make sure that it is correct: each row should have a `1` for the (unique) category you chose and `0` for all other categories.

In [ ]:
# Turn into a DataFrame
df_annotated = pd.DataFrame.from_dict({"Sentence":annotations.keys(), "Label":annotations.values()})
df_annotated['Sentence_id'] = df['Sentence_id']
# Apply one-hot-encoding (decompose label values)
for c in categories:
  df_annotated[c] = [1 if row['Label']==c else 0 for _, row in df_annotated.iterrows()]

df_annotated

Please, use a unique indentifier for the name of your annotation file.

In [ ]:
output_filename = "annotation_submission_XX" # unique identifier for your annotations

df_annotated.to_csv(f'{output_filename}.csv', index=False) # save to disk
print(f"Successfully exported {len(df_annotated)} annotations to '{output_filename}.csv'")

Your annotation file is ready!

You can now download your annotated `.csv` from the `📁 Files` folder (if running on Colab).

Once downloaded, send it to me by email so that we can group each of the individual annotations.

## Put everything together

Now that I have gathered all of your annotations, I can group them in a single DataFrame.

In [ ]:
from helpers import github_glob # helper function (in previously downloaded helpers.py) to use as glob.glob
import pandas as pd

# The annotation files your produced during the workshop are stored in the
# `data/annotation_data/your_annotations` folder on the GitHub repository
FOLDER_PATH = "data/annotation_data/your_annotations/"

# We will use a pre-defined helper function to iterate over files in
# a specific GitHub folder as if we were using the `glob.glob(<PATH>)` method in local

loaded_dfs = []
glued_data = pd.DataFrame()

for file_name in github_glob("data/annotation_data/your_annotations", "*.csv"): # equivalent to local glob.glob()
    print(file_name) # print the filename
    curr_df = load_csv_from_github(f"{FOLDER_PATH}{file_name}") # load the current csv
    loaded_dfs += [curr_df] # store the current dataframe in the list of all dataframes

glued_data = pd.concat(loaded_dfs,axis=0) # concatenate
n_student_annotators = len(loaded_dfs) # the number of annotators (used to compute agreement indices)

We can now *group* all the individual annotation files into a single one, storing the frequency of labels per category for each sample.

In [ ]:
grouped_df = glued_data[
    ["Sentence_id", "Sentence"]+categories # select columns
].groupby(
    ["Sentence_id", "Sentence"] # group on the common columns
).mean( # compute means (= frequency) for each label category
).reset_index() # reset the index

grouped_df # display the resulting DataFrame

## Compute Agreement metrics

It's time to see if you agreed or not!

Let's compute IAA metrics and compare the annotations for individual (or, most interesting) samples.

### Aggregate agreement metrics

We start by computing Fleiss' kappa and Krippendorff's alpha as metrics of overall agreement for this annotation task.

Below is a helper function to compute these quantities based on our data architecture.

In [ ]:
import numpy as np
import krippendorff
from statsmodels.stats.inter_rater import fleiss_kappa

def get_agreement(
    df_count,
    categories:list = ["Positive", "Negative", "Mixed", "Neutral"],
    n_annotators:int = None,
):
  """

  """
  # Make sure that we either: 1. know the number of annotators explicitely, or 2. have raw counts of the labels
  assert ((not n_annotators is None) or (not df_count.iloc[0][categories].sum() == 1))

  if df_count.iloc[0][categories].sum() < 2: # turn percentages into integer counts (<2 to avoid bitwise precision from ==1)
    counts = np.rint(n_annotators*df_count[categories].values).astype(int)
  else: # only change format
    counts = np.rint(df_count[categories].values).astype(int)

  # Compute Fleiss' kappa
  fleiss_k = fleiss_kappa(counts)

  # Compute Krippendorff's alpha
  ## Encode categories as integers:
  ## 0=Positive, 1=Negative, 2=Mixed, 3=Neutral
  ratings = np.array([
      [label_id for label_id, count in enumerate(row) for _ in range(count)]
      for row in counts
  ]).T

  kripp_a = krippendorff.alpha(
      reliability_data=ratings,
      level_of_measurement="nominal"
  )

  # Return Fleiss' kappa and Krippendorff's alpha as dict
  return {
      "Fleiss": fleiss_k,
      "Krippendorff": kripp_a
  }

In [ ]:
# Surprise: we also have the (detailed) annotation from the original study!
# So let's load their annotations, and we will be able to compare it with yours as well

path_to_original_annotated_df = "data/annotation_data/sentences_with_annot.csv"

df_og = load_csv_from_github(path_to_original_annotated_df)

df_og

Let's compute the agreement metrics...

In [ ]:
agreement_og = get_agreement(
    df_count=df_og,
    n_annotators=53
)

In [ ]:
agreement_students = get_agreement(
    df_count = grouped_df,
    n_annotators = n_student_annotators
)

... and print the results!

In [ ]:
print(f"Agreement Metrics")
print(f"Original Paper             : Fleiss' κ : {agreement_og['Fleiss']:.3f} | Krippendorff's α = {agreement_og['Krippendorff']:.3f} ")
print(f"Your collective annotations: Fleiss' κ = {agreement_students['Fleiss']:.3f} | Krippendorff's α = {agreement_students['Krippendorff']:.3f} ")

So, what do you think about these results?

Try to think about what these values mean, why they are similar or different, how they could be improved, if the task did even make sense, etc.

### Individual Sample Labels

In addition to summary statistics (such Fleiss' kappa and Krippendorff's alpha), we can also explore the potential disagreements at the level of individual sample.

This is a particularly important step for adjunction (determining the final label of samples)!

The function below is a helper function to visualise the proportion of each label category given to each sample.

In [ ]:
import matplotlib.pyplot as plt

def visualise_indiv_annotation_proportions(
    scores: dict,
    prop: bool = True,
    colours: dict | None = {
        "Positive": "#2E8B57",
        "Mixed":    "#E9C46A",
        "Neutral":  "#A9A9A9",
        "Negative": "#D65A4A",
    },
    title:str="",
    show_legend:bool=True
):
    if not prop:
        scores = {k: v / np.sum(list(scores.values())) for k, v in scores.items()}

    left = 0
    fig, ax = plt.subplots(figsize=(8,1.5))
    for category, value in scores.items():
        ax.barh(
            ["Annotations"],
            [value],
            left=left,
            color=colours.get(category, "#808080") if not colours is None else None,
            label=category,
            # height=0.001,
        )
        if value > 0.03:
            ax.text(
                left + value / 2,
                0,
                f"{value:.0%}",
                ha="center",
                va="center",
                color="white" if category != "Neutral" else "black",
                fontweight="bold",
            )
        left += value

    ax.set_xlim(0, 1)
    ax.set_xticks(np.linspace(0, 1, 6), [f"{x:.0%}" for x in np.linspace(0, 1, 6)])
    ax.set_yticks([])
    if show_legend:
      ax.legend(ncol=4, bbox_to_anchor=(0.5, -0.3), loc="upper center", frameon=False)
    ax.set_title(title)
    plt.box(False)
    plt.tight_layout()
    plt.show()

We can now use this function to inspect the distribution of labels for each sample:

In [ ]:
# Inspect the results

for row_id, row in df_og.iterrows():
  print("="*100)
  print(f"[{row['Sentence_type']}] (Ground Truth: {row['Correct']})\n{row['Sentence']}")
  # OG Results
  visualise_indiv_annotation_proportions(
      scores={cat: row[cat] for cat in categories},
      title = "Original",
  )

  # Students results
  visualise_indiv_annotation_proportions(
      scores={cat: grouped_df.loc[row_id][cat] for cat in categories},
      title = "Students"
  )

So, how should we determine what label to choose for samples with high disagreement?

In [ ]:
# Decompose per sentence type (label (!) in the original study)
for sent_type in df_og['Sentence_type'].unique():
  df_og_type = df_og[df_og['Sentence_type']==sent_type]

  # Do not compute agreement when less than 2 samples in the category
  if len(df_og_type) > 2:
    print("="*50)
    print(f"Sentence type: [{sent_type}] (Number of samples: {len(df_og_type)})")
    og_agg = get_agreement(
        df_count = df_og_type,
        n_annotators=53
    )
    # Get the corresponding samples with your annotations
    grouped_df_type = grouped_df.set_index('Sentence_id').loc[df_og_type['Sentence_id']].reset_index()
    students_agg = get_agreement(
        df_count = grouped_df_type,
        n_annotators=n_student_annotators
    )

    print(f"Agreement κ from original annotations = {og_agg['Fleiss']:.3f}")
    print(f"Agreement κ from your annotations     = {students_agg['Fleiss']:.3f}")